Running Inference and producing confusion matrices

In [ ]:
import os
import json
import base64
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

from openai import OpenAI

# ==========================================================
# CONFIG
# ==========================================================

DATASET_DIR = Path(
    r"Training_dataset.zip\val"
)

JSONL_PATH = DATASET_DIR / "val.jsonl"
OUTPUT_DIR = DATASET_DIR / "gpt5nano_results"
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME = "gpt-5-nano"


MAX_WORKERS = 6

SAVE_EVERY = 50

VALID_LABELS = ["non_flare", "spike", "gaussian", "gamma", "fred"]

# ==========================================================
# LOAD API KEY
# ==========================================================

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ==========================================================
# HELPERS
# ==========================================================

def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def parse_label(text):
    text = text.lower()
    for label in VALID_LABELS:
        if f"class: {label}" in text:
            return label
    for label in VALID_LABELS:
        if label in text:
            return label
    return "unknown"


SYSTEM_PROMPT = """
You are an astronomer classifying quasar light-curve morphologies.

Allowed classes:
non_flare, spike, gaussian, gamma, fred

Respond ONLY in this format:

class: <label>
shape: <short description>
confidence: <low|medium|high>
"""

# ==========================================================
# LOAD DATASET
# ==========================================================

rows = []

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        gt_text = item["messages"][1]["content"].lower()
        gt_label = parse_label(gt_text)

        img_path = DATASET_DIR / item["images"][0]

        rows.append({
            "image_path": str(img_path),
            "ground_truth": gt_label
        })

df = pd.DataFrame(rows)

print(f"Loaded {len(df)} samples")

# ==========================================================
# RESUME SUPPORT
# ==========================================================

pred_csv = OUTPUT_DIR / "predictions.csv"

if pred_csv.exists():
    old_df = pd.read_csv(pred_csv)
    done_paths = set(old_df["image_path"].tolist())
    print(f"Resuming: {len(done_paths)} already completed")

    df = df[~df["image_path"].isin(done_paths)]
    predictions = old_df.to_dict("records")
else:
    predictions = []

print(f"Remaining to process: {len(df)}")

# lock for thread-safe saving
lock = threading.Lock()

# ==========================================================
# WORKER FUNCTION
# ==========================================================

def process_row(row):

    img_path = row["image_path"]

    try:
        img_b64 = encode_image(img_path)

        response = client.responses.create(
            model=MODEL_NAME,
            input=[
                {
                    "role": "user",
                    "content": [
                        {"type": "input_text", "text": SYSTEM_PROMPT},
                        {
                            "type": "input_image",
                            "image_url": f"data:image/png;base64,{img_b64}",
                        },
                    ],
                }
            ]
        )

        output_text = response.output_text
        pred_label = parse_label(output_text)

    except Exception as e:
        output_text = f"ERROR: {e}"
        pred_label = "unknown"

    return {
        "image_path": img_path,
        "ground_truth": row["ground_truth"],
        "prediction": pred_label,
        "raw_output": output_text
    }

# ==========================================================
# PARALLEL INFERENCE
# ==========================================================

futures = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    for _, row in df.iterrows():
        futures.append(executor.submit(process_row, row))

    for i, future in enumerate(
        tqdm(as_completed(futures), total=len(futures), desc="Classifying")
    ):

        result = future.result()

        with lock:
            predictions.append(result)

            # periodic autosave
            if len(predictions) % SAVE_EVERY == 0:
                pd.DataFrame(predictions).to_csv(pred_csv, index=False)

# final save
pred_df = pd.DataFrame(predictions)
pred_df.to_csv(pred_csv, index=False)
pred_df.to_json(OUTPUT_DIR / "predictions.json", orient="records", indent=2)

print("Inference complete.")

# ==========================================================
# CONFUSION MATRIX
# ==========================================================

y_true = pred_df["ground_truth"]
y_pred = pred_df["prediction"]

cm = confusion_matrix(y_true, y_pred, labels=VALID_LABELS)

cm_df = pd.DataFrame(cm,
                     index=VALID_LABELS,
                     columns=VALID_LABELS)

cm_df.to_csv(OUTPUT_DIR / "confusion_matrix.csv")

plt.figure(figsize=(8,6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("GPT-5 nano Confusion Matrix")
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=300)
plt.show()

report = classification_report(
    y_true, y_pred, labels=VALID_LABELS, zero_division=0
)

with open(OUTPUT_DIR / "classification_report.txt", "w") as f:
    f.write(report)

print(report)
